<a href="https://colab.research.google.com/github/satria1408/saas-microservice1/blob/main/scan_buku_qwen2vl_v3_cache_isbn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Book Scanner - Qwen2-VL (3 Field Inti + Cache SQLite + Jalur ISBN)



## 1. Setup: Mount Google Drive

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

FOLDER_PATH = "/content/drive/MyDrive/google-book-scenner"
print("Folder kerja:", FOLDER_PATH)

Mounted at /content/drive
Folder kerja: /content/drive/MyDrive/google-book-scenner


## 2. Setup: Install dependencies

In [2]:
!pip install -q transformers accelerate qwen-vl-utils pillow pandas
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart requests
!pip install -q opencv-contrib-python
!pip install -q zxing-cpp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.3 MB/s eta 0:00:00


## 3. Setup: Load model Qwen2-VL-2B-Instruct

In [3]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Model siap di:", model.device)

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model siap di: cuda:0


In [ ]:
# from huggingface_hub import login
# login(token="isi_token_baru_kamu_di_sini")

## 4. Setup: Cache SQLite (2 tabel)


In [4]:
import sqlite3
import hashlib
import re

CACHE_DB_PATH = os.path.join(FOLDER_PATH, "book_cache.db")


def _hash_gambar(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def _normalisasi_key(judul: str, penulis: str) -> str:
    gabungan = f"{judul or ''}_{penulis or ''}".lower().strip()
    return re.sub(r"[^a-z0-9]+", "_", gabungan)


def _pastikan_kolom(conn, tabel: str, kolom: str, tipe: str):
    """Cek kolom sudah ada di tabel atau belum, kalau belum, tambahkan.
    Dipakai biar nambah field baru ke depannya cukup daftarin di
    DAFTAR_MIGRASI, tidak perlu tulis blok cek-ALTER manual lagi."""
    kolom_ada = [row[1] for row in conn.execute(f"PRAGMA table_info({tabel})").fetchall()]
    if kolom not in kolom_ada:
        conn.execute(f"ALTER TABLE {tabel} ADD COLUMN {kolom} {tipe}")


# Daftar semua migrasi kolom yang pernah/akan dibutuhkan, di 1 tempat.
# Kalau nanti nambah field baru (misal "edisi", "tahun"), tinggal
# tambah 1 baris di sini.
DAFTAR_MIGRASI = [
    ("cache_scan", "kategori", "TEXT"),
    ("cache_metadata", "isbn", "TEXT"),
    ("katalog", "isbn", "TEXT"),
    ("katalog", "kategori", "TEXT"),
    ("rag_manual", "isbn", "TEXT"),
]


def _init_cache_db():
    conn = sqlite3.connect(CACHE_DB_PATH)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS cache_scan (
            hash_gambar TEXT PRIMARY KEY,
            judul TEXT,
            penulis TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS cache_metadata (
            judul_penulis_key TEXT PRIMARY KEY,
            penerbit TEXT,
            sumber TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS katalog (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            judul TEXT,
            penulis TEXT,
            penerbit TEXT,
            penerbit_sumber TEXT,
            stok INTEGER DEFAULT 1,
            status_konfirmasi TEXT DEFAULT 'otomatis',
            waktu_masuk TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS rag_manual (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            judul TEXT,
            penulis TEXT,
            penerbit TEXT,
            sumber TEXT DEFAULT 'input_manual',
            waktu_masuk TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # Jalankan semua migrasi kolom yang terdaftar
    for tabel, kolom, tipe in DAFTAR_MIGRASI:
        _pastikan_kolom(conn, tabel, kolom, tipe)

    conn.commit()
    conn.close()


_init_cache_db()
print("Cache SQLite siap di:", CACHE_DB_PATH)

Cache SQLite siap di: /content/drive/MyDrive/google-book-scenner/book_cache.db


## 5. Fitur Scan: Prompt (3 field inti, versi ringkas)



In [5]:
PROMPT = """Lihat sampul buku ini dan ceritakan yang kamu lihat.

Kira-kira apa judul buku ini, siapa penulisnya, siapa penerbitnya
(nama perusahaan penerbit, bukan nama orang), dan buku ini termasuk
kategori/genre apa (misalnya: fiksi, non-fiksi, self-help, akademik,
buku anak/remaja, agama, atau lainnya)?

Kalau ada yang tidak terlihat jelas atau kamu tidak yakin, jawab null
saja untuk bagian itu - itu jawaban yang wajar, tidak masalah.

Jawab dalam format JSON:
{"judul": "...", "penulis": "...", "penerbit": "...", "kategori": "..."}"""

## 6. Fitur Scan: `ask_model` (dengan cache gambar)

.

In [6]:
import json
import re as _re
from qwen_vl_utils import process_vision_info

CANONICAL_FIELDS = ["judul", "penulis", "penerbit", "kategori"]
KATEGORI_VALID = ["fiksi", "non_fiksi", "self_help", "akademik", "anak_remaja", "agama", "lainnya"]


def _normalisasi_kategori(nilai: str) -> str:
    """Ubah jawaban bebas dari Qwen (misal 'Self-help', 'non fiksi',
    ada typo dsb) jadi salah satu kategori baku, biar konsisten dipakai
    untuk filter/browse nanti - bukan cuma tampilan yang beda-beda."""
    if not nilai:
        return None
    bersih = _re.sub(r"[^a-z]+", "_", nilai.lower()).strip("_")
    for k in KATEGORI_VALID:
        if bersih == k or bersih in k or k in bersih:
            return k
    return "lainnya"


def _normalize_keys(d: dict) -> dict:
    result = {field: None for field in CANONICAL_FIELDS}
    for raw_key, value in d.items():
        clean_key = _re.sub(r"[^a-zA-Z]", "", str(raw_key)).lower()
        for field in CANONICAL_FIELDS:
            if clean_key == field and value not in (None, ""):
                if isinstance(value, str):
                    value = value.strip().strip('"').strip()
                if field == "kategori" and isinstance(value, str):
                    result[field] = _normalisasi_kategori(value)
                else:
                    result[field] = value
    return result


def _generate_once(image_path: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=150)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


def _try_parse_scan(output_text: str):
    cleaned = output_text.strip()
    cleaned = _re.sub(r"^```json", "", cleaned).strip()
    cleaned = _re.sub(r"^```", "", cleaned).strip()
    cleaned = _re.sub(r"```$", "", cleaned).strip()
    cleaned = _re.sub(r"//[^\n]*", "", cleaned)

    match = _re.search(r"\{.*\}", cleaned, _re.DOTALL)
    if match:
        cleaned = match.group(0)

    parsed_raw = None
    try:
        parsed_raw = json.loads(cleaned)
    except json.JSONDecodeError:
        try:
            import json_repair
            parsed_raw = json_repair.loads(cleaned)
        except Exception:
            parsed_raw = None

    if not isinstance(parsed_raw, dict):
        return None

    normalized = _normalize_keys(parsed_raw)
    if not normalized.get("judul") or not normalized.get("penulis"):
        return None

    return normalized


def ask_model(image_path: str, max_retry: int = 2, lengkapi_otomatis: bool = True) -> dict:
    hash_gbr = _hash_gambar(image_path)

    conn = sqlite3.connect(CACHE_DB_PATH)
    row = conn.execute(
        "SELECT judul, penulis, kategori FROM cache_scan WHERE hash_gambar = ?", (hash_gbr,)
    ).fetchone()
    conn.close()

    if row:
        judul, penulis, kategori = row
        parsed = {"judul": judul, "penulis": penulis, "penerbit": None,
                  "kategori": kategori, "_dari_cache_scan": True}
    else:
        last_raw_output = ""
        parsed = None
        for attempt in range(max_retry + 1):
            output_text = _generate_once(image_path)
            last_raw_output = output_text
            hasil = _try_parse_scan(output_text)
            if hasil is not None:
                hasil["_attempt"] = attempt + 1
                hasil["_dari_cache_scan"] = False
                parsed = hasil
                break

        if parsed is None:
            return {"judul": None, "penulis": None, "penerbit": None, "kategori": None,
                    "_raw_output": last_raw_output, "_parse_error": True,
                    "status_konfirmasi": "gagal_scan"}

        conn = sqlite3.connect(CACHE_DB_PATH)
        conn.execute(
            "INSERT OR REPLACE INTO cache_scan (hash_gambar, judul, penulis, kategori) VALUES (?, ?, ?, ?)",
            (hash_gbr, parsed.get("judul"), parsed.get("penulis"), parsed.get("kategori")),
        )
        conn.commit()
        conn.close()

    parsed["_penerbit_tebakan_vlm"] = parsed.get("penerbit")

    if lengkapi_otomatis:
        hasil_meta = lengkapi_metadata(parsed.get("judul"), parsed.get("penulis"))
        parsed["penerbit"] = hasil_meta.get("penerbit")
        parsed["_penerbit_sumber"] = hasil_meta.get("sumber")
        parsed["isbn"] = hasil_meta.get("isbn")
    else:
        parsed["_penerbit_sumber"] = "tidak_dicek"

    parsed["status_konfirmasi"] = "otomatis"
    parsed["_id_katalog"] = _simpan_atau_gabung_katalog(parsed)
    return parsed

## 7. Fitur Scan: Lengkapi Penerbit - Open Library (dengan cache judul)



In [7]:
import requests
import difflib


def _ambil_dari_open_library(judul: str, penulis: str):
    headers = {"User-Agent": "ScanBukuPerpus/1.0 (contoh@sekolah.sch.id)"}
    BAHASA_PRIORITAS = {"eng", "ind"}

    query = judul
    if penulis:
        query += f" {penulis}"
    search_url = (
        f"https://openlibrary.org/search.json?q={requests.utils.quote(query)}"
        f"&fields=title,author_name,publisher,key&limit=5"
    )

    try:
        res = requests.get(search_url, headers=headers, timeout=15)
        if res.status_code != 200:
            print(f"[DEBUG] Open Library status {res.status_code}")
            return None
        docs = res.json().get("docs", [])
    except Exception as e:
        print(f"[DEBUG] Open Library error: {e}")
        return None

    if not docs:
        return None

    work_key = docs[0].get("key")
    publisher_agregat_fallback = None
    if docs[0].get("publisher"):
        publisher_agregat_fallback = docs[0]["publisher"][0]

    if not work_key:
        return publisher_agregat_fallback

    try:
        ed_url = f"https://openlibrary.org{work_key}/editions.json?limit=20"
        res = requests.get(ed_url, headers=headers, timeout=15)
        if res.status_code != 200:
            return publisher_agregat_fallback
        entries = res.json().get("entries", [])
    except Exception as e:
        print(f"[DEBUG] Open Library editions error: {e}")
        return publisher_agregat_fallback

    kandidat_lain = []
    for ed in entries:
        pub = ed.get("publishers")
        if not pub:
            continue
        lang_keys = [l.get("key", "").split("/")[-1] for l in ed.get("languages", [])]
        if any(l in BAHASA_PRIORITAS for l in lang_keys):
            return pub[0]
        kandidat_lain.append(pub[0])

    if kandidat_lain:
        return kandidat_lain[0]

    if publisher_agregat_fallback:
        print(f"[DEBUG] Tidak ada edisi eng/ind dengan publisher untuk {work_key}, pakai fallback agregat")
    return publisher_agregat_fallback


def tambah_ke_rag_manual(judul: str, penulis: str, penerbit: str, isbn: str = None, sumber: str = "input_manual"):
    """Simpan/update data buku yang sudah diverifikasi manual ke basis
    RAG - dipakai sebagai override sebelum Open Library ditanya."""
    with sqlite3.connect(CACHE_DB_PATH) as conn:
        sudah_ada = conn.execute(
            "SELECT id FROM rag_manual WHERE LOWER(judul) = LOWER(?) AND LOWER(penulis) = LOWER(?)",
            (judul, penulis)
        ).fetchone()

        if sudah_ada:
            conn.execute(
                "UPDATE rag_manual SET penerbit = ?, isbn = ?, sumber = ? WHERE id = ?",
                (penerbit, isbn, sumber, sudah_ada[0])
            )
            print(f"Diupdate di RAG manual: {judul} - {penerbit} - ISBN: {isbn}")
        else:
            conn.execute(
                "INSERT INTO rag_manual (judul, penulis, penerbit, isbn, sumber) VALUES (?, ?, ?, ?, ?)",
                (judul, penulis, penerbit, isbn, sumber),
            )
            print(f"Tersimpan ke RAG manual: {judul} - {penerbit} - ISBN: {isbn}")
        conn.commit()


def cari_di_rag_manual(judul: str, penulis: str, ambang_batas: float = 0.75):
    """Cari buku yang MIRIP di basis RAG manual, balikin dict
    {penerbit, isbn} - bukan cuma string penerbit."""
    with sqlite3.connect(CACHE_DB_PATH) as conn:
        semua = conn.execute("SELECT judul, penulis, penerbit, isbn FROM rag_manual").fetchall()

    if not semua:
        return None

    query_gabungan = f"{judul or ''} {penulis or ''}".lower()

    kandidat_terbaik = None
    skor_terbaik = 0.0

    for j, p, penerbit, isbn in semua:
        target_gabungan = f"{j or ''} {p or ''}".lower()
        skor = difflib.SequenceMatcher(None, query_gabungan, target_gabungan).ratio()
        if skor > skor_terbaik:
            skor_terbaik = skor
            kandidat_terbaik = {"penerbit": penerbit, "isbn": isbn}

    if skor_terbaik >= ambang_batas:
        return kandidat_terbaik
    return None


def lengkapi_metadata(judul: str, penulis: str) -> dict:
    if not judul:
        return {"penerbit": None, "sumber": None, "isbn": None}

    key = _normalisasi_key(judul, penulis)

    with sqlite3.connect(CACHE_DB_PATH) as conn:
        row = conn.execute(
            "SELECT penerbit, sumber, isbn FROM cache_metadata WHERE judul_penulis_key = ?", (key,)
        ).fetchone()

        if row and row[0]:
            isbn = row[2]
            if not isbn:
                baris_katalog = conn.execute(
                    "SELECT isbn FROM katalog WHERE judul = ? AND penulis = ? AND isbn IS NOT NULL LIMIT 1",
                    (judul, penulis)
                ).fetchone()
                isbn = baris_katalog[0] if baris_katalog else None
            return {"penerbit": row[0], "sumber": row[1] + "_cache", "isbn": isbn}

    hasil_rag = cari_di_rag_manual(judul, penulis)
    if hasil_rag:
        penerbit = hasil_rag["penerbit"]
        isbn_rag = hasil_rag["isbn"]
        sumber = "rag_manual"
    else:
        penerbit = _ambil_dari_open_library(judul, penulis or "")
        isbn_rag = None
        sumber = "open_library" if penerbit else None

    if penerbit:
        with sqlite3.connect(CACHE_DB_PATH) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO cache_metadata (judul_penulis_key, penerbit, sumber, isbn) VALUES (?, ?, ?, ?)",
                (key, penerbit, sumber, isbn_rag),
            )
            conn.commit()

    return {"penerbit": penerbit, "sumber": sumber, "isbn": isbn_rag}


def _simpan_ke_katalog(hasil: dict) -> int:
    conn = sqlite3.connect(CACHE_DB_PATH)
    cursor = conn.execute(
        """INSERT INTO katalog (judul, penulis, penerbit, penerbit_sumber, isbn, kategori, status_konfirmasi)
           VALUES (?, ?, ?, ?, ?, ?, ?)""",
        (hasil.get("judul"), hasil.get("penulis"), hasil.get("penerbit"),
         hasil.get("_penerbit_sumber"), hasil.get("isbn"), hasil.get("kategori"), "otomatis"),
    )
    conn.commit()
    id_baru = cursor.lastrowid
    conn.close()
    return id_baru


def _bandingkan_penerbit(a: str, b: str) -> bool:
    def bersih(s):
        s = (s or "").strip().lower().replace("&", "and")
        return re.sub(r"[^a-z0-9]+", " ", s).strip()
    a_bersih, b_bersih = bersih(a), bersih(b)
    if a_bersih == b_bersih:
        return True
    if a_bersih and b_bersih:
        return a_bersih in b_bersih or b_bersih in a_bersih
    return False


def _simpan_atau_gabung_katalog(hasil: dict) -> int:
    key_baru = _normalisasi_key(hasil.get("judul"), hasil.get("penulis"))

    conn = sqlite3.connect(CACHE_DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("SELECT id, judul, penulis, penerbit, isbn FROM katalog").fetchall()

    for row in rows:
        if _normalisasi_key(row["judul"], row["penulis"]) != key_baru:
            continue

        lama = row["penerbit"]
        baru = hasil.get("penerbit")
        cocok = (not lama) or (not baru) or _bandingkan_penerbit(lama, baru)

        if cocok:
            if hasil.get("isbn") and not row["isbn"]:
                conn.execute("UPDATE katalog SET isbn = ? WHERE id = ?", (hasil["isbn"], row["id"]))
            if hasil.get("penerbit") and not row["penerbit"]:
                conn.execute("UPDATE katalog SET penerbit = ? WHERE id = ?", (hasil["penerbit"], row["id"]))
            conn.execute("UPDATE katalog SET stok = stok + 1 WHERE id = ?", (row["id"],))
            conn.commit()
            conn.close()
            return row["id"]

    conn.close()
    return _simpan_ke_katalog(hasil)


def lihat_katalog(hanya_belum_dikonfirmasi: bool = False) -> list:
    conn = sqlite3.connect(CACHE_DB_PATH)
    conn.row_factory = sqlite3.Row
    query = "SELECT * FROM katalog"
    if hanya_belum_dikonfirmasi:
        query += " WHERE status_konfirmasi = 'otomatis'"
    rows = conn.execute(query).fetchall()
    conn.close()
    return [dict(r) for r in rows]

In [8]:
import pandas as pd
df_katalog = pd.DataFrame(lihat_katalog())
df_katalog

,id,judul,penulis,penerbit,penerbit_sumber,stok,status_konfirmasi,waktu_masuk,isbn,kategori
0,38,Metamorphosis and Other Stories,Franz Kafka,Arcturus Publishing Limited,open_library,1,otomatis,2026-09-23 04:02:48,None,fiksi
1,39,Of Mice and Men,John Steinbeck,Collier,open_library,1,otomatis,2026-09-23 04:09:20,None,fiksi
2,40,Things Fall Apart,Chinua Achebe,Heinemann,open_library,1,otomatis,2026-09-23 04:32:57,None,fiksi
3,41,Student Hidjo,Mas Marco Kartodikromo,Narasi,rag_manual,1,otomatis,2026-09-23 04:42:43,9786237586616,None


In [ ]:
conn = sqlite3.connect(CACHE_DB_PATH)
df_rag = pd.read_sql_query("SELECT * FROM rag_manual", conn)
df_cache = pd.read_sql_query("SELECT * FROM cache_metadata WHERE sumber = 'rag_manual'", conn)
conn.close()

print("Total buku di RAG manual:", len(df_rag))
print("Berapa yang sudah pernah 'kepakai' (ada di cache_metadata):", len(df_cache))

print("Persentase ISBN terisi:", df_rag["isbn"].notna().sum(), "/", len(df_rag))

In [17]:
import pandas as pd

conn = sqlite3.connect(CACHE_DB_PATH)
df_rag = pd.read_sql_query("SELECT * FROM rag_manual", conn)
conn.close()

df_rag

,id,judul,penulis,penerbit,sumber,waktu_masuk,isbn
0,1,Student Hidjo,Mas Marco Kartodikromo,Narasi,input_manual,2026-09-18 10:26:24,9786237586616
1,2,Azab dan Sengsara,Merari Siregar,Balai Pustaka,input_manual,2026-09-18 10:26:24,9789794071687
2,3,Perca,Syhrn Azr,Ellunar Publisher,input_manual,2026-09-18 10:26:24,9786232041905
3,4,Reuni,Ratna Destiari,Ellunar Publisher,input_manual,2026-09-18 10:26:24,9786233851787
4,5,Kisah Romantis Tingkat Baper,Susi,Ellunar Publisher,input_manual,2026-09-18 10:26:24,9786232040311
...,...,...,...,...,...,...,...
207,208,Blink,Malcolm Gladwell,Gramedia Pustaka Utama,input_manual,2026-09-23 09:09:12,None
208,209,The Tipping Point,Malcolm Gladwell,Gramedia Pustaka Utama,input_manual,2026-09-23 09:09:12,None
209,210,Sitti Nurbaya,Marah Rusli,Balai Pustaka,input_manual,2026-09-23 09:09:12,None
210,211,Salah Asuhan,Abdul Muis,Balai Pustaka,input_manual,2026-09-23 09:09:12,None


In [16]:
tambah_ke_rag_manual("Gadis Pantai", "Pramoedya Ananta Toer", "Hasta Mitra", None)
tambah_ke_rag_manual("Bukan Pasar Malam", "Pramoedya Ananta Toer", "Hasta Mitra", None)
tambah_ke_rag_manual("Ronggeng Dukuh Paruk", "Ahmad Tohari", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Lintang Kemukus Dini Hari", "Ahmad Tohari", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Jantera Bianglala", "Ahmad Tohari", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Bekisar Merah", "Ahmad Tohari", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Kubah", "Ahmad Tohari", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Orang-Orang Proyek", "Ahmad Tohari", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Lelaki Harimau", "Eka Kurniawan", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Seperti Dendam, Rindu Harus Dibayar Tuntas", "Eka Kurniawan", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("O", "Eka Kurniawan", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Perempuan Patah Hati yang Kembali Menemukan Cinta Melalui Mimpi", "Eka Kurniawan", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Amba", "Laksmi Pamuntjak", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Aruna dan Lidahnya", "Laksmi Pamuntjak", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Bilangan Fu", "Ayu Utami", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Larung", "Ayu Utami", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Aroma Karsa", "Dee Lestari", "Bentang Pustaka", None)
tambah_ke_rag_manual("Supernova: Ksatria, Puteri dan Bintang Jatuh", "Dee Lestari", "Bentang Pustaka", None)
tambah_ke_rag_manual("Supernova: Akar", "Dee Lestari", "Bentang Pustaka", None)
tambah_ke_rag_manual("Supernova: Petir", "Dee Lestari", "Bentang Pustaka", None)
tambah_ke_rag_manual("Filosofi Kopi", "Dee Lestari", "Bentang Pustaka", None)
tambah_ke_rag_manual("Rapijali 1: Mencari", "Dee Lestari", "Bentang Pustaka", None)
tambah_ke_rag_manual("Milea: Suara dari Dilan", "Pidi Baiq", "Pastel Books", None)
tambah_ke_rag_manual("Ancika: Dia yang Bersamaku Tahun 1995", "Pidi Baiq", "Pastel Books", None)
tambah_ke_rag_manual("Kambing Jantan", "Raditya Dika", "GagasMedia", None)
tambah_ke_rag_manual("Manusia Setengah Salmon", "Raditya Dika", "GagasMedia", None)
tambah_ke_rag_manual("Marmut Merah Jambu", "Raditya Dika", "Bukune", None)
tambah_ke_rag_manual("Nanti Kita Cerita tentang Hari Ini", "Marchella F.P.", "POP", None)
tambah_ke_rag_manual("Geez & Ann", "Rintik Sedu", "GagasMedia", None)
tambah_ke_rag_manual("Kata", "Rintik Sedu", "GagasMedia", None)
tambah_ke_rag_manual("Kita Pergi Hari Ini", "Ziggy Zezsyazeoviennazabrizkie", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Jakarta Sebelum Pagi", "Ziggy Zezsyazeoviennazabrizkie", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Nadira", "Leila S. Chudori", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Sebuah Seni untuk Bersikap Bodo Amat", "Mark Manson", "Gramedia Widiasarana Indonesia", None)
tambah_ke_rag_manual("The Almanack of Naval Ravikant", "Eric Jorgenson", "Elex Media Komputindo", None)
tambah_ke_rag_manual("Rich Dad Poor Dad", "Robert T. Kiyosaki", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Norwegian Wood", "Haruki Murakami", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Kafka on the Shore", "Haruki Murakami", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("1Q84", "Haruki Murakami", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Colorless Tsukuru Tazaki and His Years of Pilgrimage", "Haruki Murakami", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("The Alchemist", "Paulo Coelho", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Veronika Memutuskan Mati", "Paulo Coelho", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Brida", "Paulo Coelho", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Eleven Minutes", "Paulo Coelho", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Kite Runner", "Khaled Hosseini", "Qanita", None)
tambah_ke_rag_manual("A Thousand Splendid Suns", "Khaled Hosseini", "Qanita", None)
tambah_ke_rag_manual("The Little Prince", "Antoine de Saint-Exupéry", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Animal Farm", "George Orwell", "Bentang Pustaka", None)
tambah_ke_rag_manual("1984", "George Orwell", "Bentang Pustaka", None)
tambah_ke_rag_manual("Pride and Prejudice", "Jane Austen", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Jane Eyre", "Charlotte Brontë", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Wuthering Heights", "Emily Brontë", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Great Gatsby", "F. Scott Fitzgerald", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("To Kill a Mockingbird", "Harper Lee", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Catcher in the Rye", "J. D. Salinger", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Old Man and the Sea", "Ernest Hemingway", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Stranger", "Albert Camus", "Yayasan Pustaka Obor Indonesia", None)
tambah_ke_rag_manual("The Trial", "Franz Kafka", "Bentang Pustaka", None)
tambah_ke_rag_manual("Things Fall Apart", "Chinua Achebe", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Picture of Dorian Gray", "Oscar Wilde", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("A Room of One's Own", "Virginia Woolf", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Harry Potter dan Batu Bertuah", "J. K. Rowling", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Harry Potter dan Kamar Rahasia", "J. K. Rowling", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Harry Potter dan Tawanan Azkaban", "J. K. Rowling", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Harry Potter dan Piala Api", "J. K. Rowling", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Harry Potter dan Orde Phoenix", "J. K. Rowling", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Harry Potter dan Pangeran Berdarah Campuran", "J. K. Rowling", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Harry Potter dan Relikui Kematian", "J. K. Rowling", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Hunger Games", "Suzanne Collins", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Catching Fire", "Suzanne Collins", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Mockingjay", "Suzanne Collins", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Fault in Our Stars", "John Green", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Looking for Alaska", "John Green", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Perks of Being a Wallflower", "Stephen Chbosky", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Wonder", "R. J. Palacio", "Noura Books", None)
tambah_ke_rag_manual("The Book Thief", "Markus Zusak", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Help", "Kathryn Stockett", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Me Before You", "Jojo Moyes", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Midnight Library", "Matt Haig", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Eleanor Oliphant Is Completely Fine", "Gail Honeyman", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Educated", "Tara Westover", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Becoming", "Michelle Obama", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Sapiens", "Yuval Noah Harari", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Homo Deus", "Yuval Noah Harari", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("21 Lessons for the 21st Century", "Yuval Noah Harari", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("A Brief History of Time", "Stephen Hawking", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Cosmos", "Carl Sagan", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Guns, Germs, and Steel", "Jared Diamond", "Kepustakaan Populer Gramedia", None)
tambah_ke_rag_manual("Quiet", "Susan Cain", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Thinking, Fast and Slow", "Daniel Kahneman", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Outliers", "Malcolm Gladwell", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Blink", "Malcolm Gladwell", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("The Tipping Point", "Malcolm Gladwell", "Gramedia Pustaka Utama", None)
tambah_ke_rag_manual("Sitti Nurbaya", "Marah Rusli", "Balai Pustaka", None)
tambah_ke_rag_manual("Salah Asuhan", "Abdul Muis", "Balai Pustaka", None)
tambah_ke_rag_manual("Layar Terkembang", "Sutan Takdir Alisjahbana", "Balai Pustaka", None)

Tersimpan ke RAG manual: Gadis Pantai - Hasta Mitra - ISBN: None
Diupdate di RAG manual: Bukan Pasar Malam - Hasta Mitra - ISBN: None
Tersimpan ke RAG manual: Ronggeng Dukuh Paruk - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Lintang Kemukus Dini Hari - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Jantera Bianglala - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Bekisar Merah - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Kubah - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Orang-Orang Proyek - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Lelaki Harimau - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Seperti Dendam, Rindu Harus Dibayar Tuntas - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: O - Gramedia Pustaka Utama - ISBN: None
Tersimpan ke RAG manual: Perempuan Patah Hati yang Kembali Menemukan Cinta Melalui Mimpi - Gramedia Pustaka Utama - ISBN: None
Tersimpa

In [ ]:
conn = sqlite3.connect(CACHE_DB_PATH)
conn.execute("DELETE FROM cache_scan")
conn.execute("DELETE FROM cache_metadata")
conn.execute("DELETE FROM katalog")
conn.commit()
conn.close()

## 8. Jalur ISBN (tanpa Qwen sama sekali)



In [11]:
def _cari_isbn_di_rag_manual(isbn: str):
    """Cari EXACT MATCH berdasarkan ISBN di rag_manual - lebih presisi
    dari fuzzy judul, karena ISBN itu unik. Dipakai sebagai fallback
    kalau Open Library gagal/tidak punya data buku lokal ini."""
    with sqlite3.connect(CACHE_DB_PATH) as conn:
        row = conn.execute(
            "SELECT judul, penulis, penerbit FROM rag_manual WHERE isbn = ? LIMIT 1", (isbn,)
        ).fetchone()

    if row:
        return {"judul": row[0], "penulis": row[1], "penerbit": row[2]}
    return None


def cari_dari_isbn(isbn: str) -> dict:
    isbn_bersih = isbn.strip().replace("-", "").replace(" ", "")

    with sqlite3.connect(CACHE_DB_PATH) as conn:
        row = conn.execute(
            "SELECT penerbit FROM cache_metadata WHERE isbn = ? LIMIT 1", (isbn_bersih,)
        ).fetchone()

    if row:
        with sqlite3.connect(CACHE_DB_PATH) as conn:
            baris = conn.execute(
                "SELECT judul, penulis FROM katalog WHERE isbn = ? LIMIT 1", (isbn_bersih,)
            ).fetchone()
        if baris:
            judul_asli, penulis_asli = baris
            return {
                "status": "berhasil",
                "judul": judul_asli,
                "penulis": penulis_asli,
                "penerbit": row[0],
                "isbn": isbn_bersih,
                "_penerbit_sumber": "open_library_isbn_cache",
                "status_konfirmasi": "otomatis",
            }

    headers = {"User-Agent": "ScanBukuPerpus/1.0 (contoh@sekolah.sch.id)"}
    url = f"https://openlibrary.org/api/books?bibkeys=ISBN:{isbn_bersih}&format=json&jscmd=data"

    try:
        res = requests.get(url, headers=headers, timeout=15)
        data = res.json().get(f"ISBN:{isbn_bersih}") if res.status_code == 200 else None
    except Exception as e:
        print(f"[DEBUG] Open Library ISBN error: {e}")
        data = None

    if not data:
        hasil_rag = _cari_isbn_di_rag_manual(isbn_bersih)
        if hasil_rag:
            hasil = {
                "status": "berhasil",
                "judul": hasil_rag["judul"],
                "penulis": hasil_rag["penulis"],
                "penerbit": hasil_rag["penerbit"],
                "isbn": isbn_bersih,
                "_penerbit_sumber": "rag_manual",
            }
            hasil["status_konfirmasi"] = "otomatis"
            hasil["_id_katalog"] = _simpan_atau_gabung_katalog(hasil)
            return hasil

        return {"status": "isbn_tidak_ketemu", "isbn": isbn_bersih}

    hasil = {
        "status": "berhasil",
        "judul": data.get("title"),
        "penulis": ", ".join(a.get("name", "") for a in data.get("authors", [])),
        "penerbit": ", ".join(p.get("name", "") for p in data.get("publishers", [])) or None,
        "isbn": isbn_bersih,
        "_penerbit_sumber": "open_library_isbn",
    }

    hasil["status_konfirmasi"] = "otomatis"
    hasil["_id_katalog"] = _simpan_atau_gabung_katalog(hasil)

    with sqlite3.connect(CACHE_DB_PATH) as conn:
        row_katalog = conn.execute(
            "SELECT judul, penulis FROM katalog WHERE id = ?", (hasil["_id_katalog"],)
        ).fetchone()
        judul_final, penulis_final = row_katalog if row_katalog else (hasil["judul"], hasil["penulis"])
        key = _normalisasi_key(judul_final, penulis_final)
        conn.execute(
            "INSERT OR REPLACE INTO cache_metadata (judul_penulis_key, penerbit, sumber, isbn) VALUES (?, ?, ?, ?)",
            (key, hasil.get("penerbit"), "open_library_isbn", hasil.get("isbn")),
        )
        conn.commit()

    return hasil

In [12]:
import cv2
import numpy as np
from PIL import Image, ImageOps

try:
    import zxingcpp as _zxingcpp
except ImportError:
    _zxingcpp = None

_BARCODE_DETECTOR = cv2.barcode.BarcodeDetector()


def _checksum_ean13_valid(kode: str) -> bool:
    """Validasi matematis EAN-13 (dipakai ISBN-13). Ini filter utama, BUKAN
    nama format dari library (string format beda-beda antar versi
    OpenCV/zxing-cpp, jadi rapuh kalau diandalkan) - checksum lebih pasti."""
    if len(kode) != 13 or not kode.isdigit():
        return False
    if not kode.startswith(("978", "979")):
        return False
    jumlah = sum(int(d) * (1 if i % 2 == 0 else 3) for i, d in enumerate(kode[:12]))
    return (10 - jumlah % 10) % 10 == int(kode[12])


def _kandidat_opencv(img) -> list:
    """Ambil SEMUA teks yang kedetek OpenCV di gambar ini, belum difilter."""
    try:
        retval, decoded_info, decoded_type, points = _BARCODE_DETECTOR.detectAndDecodeWithType(img)
        return [t.strip() for t in decoded_info if t]
    except Exception:
        retval, points, straight = _BARCODE_DETECTOR.detectAndDecode(img)
        kode = (retval or "").strip()
        return [kode] if kode else []


def _kandidat_zxing(img) -> list:
    """Fallback kalau OpenCV tidak menghasilkan kandidat valid."""
    if _zxingcpp is None:
        return []
    hasil = _zxingcpp.read_barcodes(img)
    return [b.text.strip() for b in hasil if b.text]


def _coba_decode(img) -> str | None:
    """OpenCV dulu -> zxing-cpp sebagai fallback. Cuma diterima kandidat
    yang lolos checksum EAN-13 dengan prefix 978/979."""
    for kandidat in (_kandidat_opencv(img) + _kandidat_zxing(img)):
        if _checksum_ean13_valid(kandidat):
            return kandidat
    return None


def _baca_gambar(image_path: str):
    """PIL dulu, baru convert ke array OpenCV - cv2.imread() gagal diam-diam
    untuk beberapa format seperti .webp. exif_transpose membetulkan
    orientasi foto HP yang sering 'miring' lewat tag EXIF."""
    try:
        img_pil = ImageOps.exif_transpose(Image.open(image_path)).convert("RGB")
    except Exception:
        return None
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


def _decode_isbn_dari_gambar(image_path: str) -> str | None:
    """Deteksi & decode ISBN dari foto. Coba full gambar dulu; kalau gagal,
    coba beberapa area crop (barcode biasanya di bawah cover belakang)
    lalu diperbesar, baru decode ulang."""
    img = _baca_gambar(image_path)
    if img is None:
        return None

    kode = _coba_decode(img)
    if kode:
        return kode

    tinggi, lebar = img.shape[:2]
    area_kandidat = [
        img[int(tinggi * 0.5):, :],
        img[int(tinggi * 0.55):, int(lebar * 0.0):int(lebar * 0.6)],
        img[int(tinggi * 0.55):, int(lebar * 0.4):],
        img[int(tinggi * 0.3):, :],
    ]

    for crop in area_kandidat:
        if crop.size == 0:
            continue
        crop_besar = cv2.resize(crop, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
        kode = _coba_decode(crop_besar)
        if kode:
            return kode

    return None


def cari_dari_foto_barcode(image_path: str) -> dict:
    """Endpoint /scan-barcode manggil ini. Foto -> ISBN (OpenCV -> zxing-cpp,
    keduanya divalidasi checksum) -> cari_dari_isbn. Qwen tidak dilibatkan."""
    isbn_terdeteksi = _decode_isbn_dari_gambar(image_path)

    if not isbn_terdeteksi:
        return {
            "status": "barcode_tidak_terdeteksi",
            "isbn": None,
            "detail": "Tidak ada barcode ISBN valid yang terbaca. Foto lebih "
                      "dekat & lurus ke barcode-nya (5-10 cm), lepas plastik "
                      "pembungkus kalau ada, atau ketik ISBN manual.",
        }

    hasil = cari_dari_isbn(isbn_terdeteksi)
    hasil["_isbn_dari_barcode"] = isbn_terdeteksi
    return hasil


# --- pengecekan otomatis, langsung kelihatan begitu cell ini di-run ---
print("=== Cek definisi ===")
for _nama in ["_checksum_ean13_valid", "_coba_decode", "_decode_isbn_dari_gambar", "cari_dari_foto_barcode"]:
    print(f"  {_nama:28s} -> {'ADA' if _nama in globals() else 'BELUM ADA'}")
print("cari_dari_isbn tersedia    ->", "ADA" if "cari_dari_isbn" in globals() else "BELUM ADA (jalankan cell cari_dari_isbn dulu!)")

=== Cek definisi ===
  _checksum_ean13_valid        -> ADA
  _coba_decode                 -> ADA
  _decode_isbn_dari_gambar     -> ADA
  cari_dari_foto_barcode       -> ADA
cari_dari_isbn tersedia    -> ADA


## 9. Konfirmasi Petugas (opsional - untuk koreksi data)



In [13]:
def update_status_konfirmasi(id_katalog: int, koreksi: dict = None, stok: int = None) -> dict:
    """
    Update entry yang SUDAH ada di katalog (bukan insert baru - insert
    sudah terjadi otomatis dari ask_model). Dipakai petugas untuk
    mengoreksi field yang salah, atau menandai sudah direview.

    id_katalog : ambil dari hasil["_id_katalog"] setelah ask_model()
    koreksi    : dict field yang mau ditimpa, misal {"penerbit": "Gramedia"}
    stok       : jumlah eksemplar fisik buku ini
    """
    conn = sqlite3.connect(CACHE_DB_PATH)

    if koreksi:
        for field, value in koreksi.items():
            if field in ("judul", "penulis", "penerbit"):
                conn.execute(f"UPDATE katalog SET {field} = ? WHERE id = ?", (value, id_katalog))

    if stok is not None:
        conn.execute("UPDATE katalog SET stok = ? WHERE id = ?", (stok, id_katalog))

    conn.execute("UPDATE katalog SET status_konfirmasi = ? WHERE id = ?", ("terkonfirmasi", id_katalog))
    conn.commit()
    conn.close()

    print(f"Katalog id={id_katalog} sudah diupdate & ditandai terkonfirmasi.")


# Contoh pakai jalur scan:
# hasil = ask_model("/path/ke/cover.webp")
# print(hasil)  # sudah otomatis masuk katalog, cek hasil["_id_katalog"]
# update_status_konfirmasi(hasil["_id_katalog"], koreksi={"penerbit": "Gramedia"}, stok=3)

# Contoh pakai jalur ISBN (perlu simpan manual karena cari_dari_isbn
# belum otomatis masuk katalog seperti ask_model):
# hasil = cari_dari_isbn("9780385533225")
# id_baru = _simpan_ke_katalog(hasil)
# update_status_konfirmasi(id_baru, stok=2)

In [ ]:
   conn = sqlite3.connect(CACHE_DB_PATH)
   conn.execute("DELETE FROM cache_metadata")  # hapus semua, mulai bersih dari nol
   conn.commit()
   conn.close()

## 10. Live API (opsional)


In [ ]:
from fastapi import FastAPI, UploadFile, File, Header, HTTPException, Request
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import shutil
import time
import os
import asyncio
from collections import defaultdict
from PIL import Image, ImageOps
import io

NGROK_AUTHTOKEN = ""
API_KEY = ""
ngrok.set_auth_token(NGROK_AUTHTOKEN)

app = FastAPI()

RATE_LIMIT_MAX = 30
RATE_LIMIT_WINDOW = 60

_riwayat_request = defaultdict(list)  # {ip: [timestamp1, timestamp2, ...]}


def _cek_rate_limit(ip: str) -> bool:
    """True kalau MASIH BOLEH request, False kalau sudah kelebihan limit."""
    sekarang = time.time()
    _riwayat_request[ip] = [t for t in _riwayat_request[ip] if sekarang - t < RATE_LIMIT_WINDOW]

    if len(_riwayat_request[ip]) >= RATE_LIMIT_MAX:
        return False

    _riwayat_request[ip].append(sekarang)
    return True


def _resize_dan_konversi(file_bytes: bytes, max_size: int = 1280) -> str:
    img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    lebar, tinggi = img.size
    if max(lebar, tinggi) > max_size:
        rasio = max_size / max(lebar, tinggi)
        img = img.resize((int(lebar * rasio), int(tinggi * rasio)), Image.LANCZOS)
    temp_path = f"/tmp/scan_{hash(file_bytes)}.jpg"
    img.save(temp_path, "JPEG", quality=88)
    return temp_path


def _simpan_untuk_barcode(file_bytes: bytes, max_size: int = 2400) -> str:
    """Beda dari _resize_dan_konversi (1280px, buat Qwen): barcode butuh resolusi
    lebih tinggi karena garisnya tipis, dan orientasi EXIF harus dibetulkan dulu
    karena foto HP sering disimpan 'miring' lewat tag EXIF."""
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(file_bytes))).convert("RGB")
    lebar, tinggi = img.size
    if max(lebar, tinggi) > max_size:
        rasio = max_size / max(lebar, tinggi)
        img = img.resize((int(lebar * rasio), int(tinggi * rasio)), Image.LANCZOS)
    temp_path = f"/tmp/barcode_{hash(file_bytes)}.jpg"
    img.save(temp_path, "JPEG", quality=95)
    return temp_path


@app.post("/scan")
async def scan_endpoint(request: Request, file: UploadFile = File(...), x_api_key: str = Header(None)):
    ip_pengirim = request.client.host
    if not _cek_rate_limit(ip_pengirim):
        raise HTTPException(status_code=429, detail="Terlalu banyak request, coba lagi sebentar")

    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="API key salah")

    file_bytes = await file.read()
    temp_path = _resize_dan_konversi(file_bytes)

    hasil = ask_model(temp_path)
    os.remove(temp_path)
    return hasil


@app.get("/isbn/{isbn}")
async def isbn_endpoint(isbn: str, request: Request, x_api_key: str = Header(None)):
    ip_pengirim = request.client.host
    if not _cek_rate_limit(ip_pengirim):
        raise HTTPException(status_code=429, detail="Terlalu banyak request, coba lagi sebentar")

    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="API key salah")
    return cari_dari_isbn(isbn)


@app.post("/scan-barcode")
async def scan_barcode_endpoint(request: Request, file: UploadFile = File(...), x_api_key: str = Header(None)):
    ip_pengirim = request.client.host
    if not _cek_rate_limit(ip_pengirim):
        raise HTTPException(status_code=429, detail="Terlalu banyak request, coba lagi sebentar")

    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="API key salah")

    file_bytes = await file.read()
    try:
        temp_path = _simpan_untuk_barcode(file_bytes)
    except Exception:
        raise HTTPException(status_code=400, detail="File bukan gambar yang valid")

    try:
        # thread terpisah: decode barcode + panggilan Open Library sifatnya blocking,
        # jangan sampai menahan event loop server
        return await asyncio.to_thread(cari_dari_foto_barcode, temp_path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


nest_asyncio.apply()
public_url = ngrok.connect(8000, domain="closable-bullish-showman.ngrok-free.dev")
print("Live API jalan di:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Live API jalan di: NgrokTunnel: "https://closable-bullish-showman.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [1785]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     103.84.230.119:0 - "GET /isbn/9786237586616 HTTP/1.1" 200 OK


In [15]:
import secrets; secrets.token_urlsafe(32)


'vJPwQUeEjP4kBJPtvrXeZKQRMNJ6dBQP2ihwVNufhpE'

In [21]:
import requests

# 1. Cek endpoint yang beda (search.json) - yang dipakai jalur scan foto
res1 = requests.get("https://openlibrary.org/search.json?q=harry+potter&limit=1", timeout=15)
print("search.json status:", res1.status_code)
print("search.json awal:", res1.text[:200])

# 2. Cek endpoint bibkeys tanpa header custom, lihat redirect/url akhir
res2 = requests.get("https://openlibrary.org/api/books?bibkeys=ISBN:9780140328721&format=json&jscmd=data", timeout=15)
print("bibkeys status:", res2.status_code)
print("bibkeys url akhir:", res2.url)
print("bibkeys history (redirect):", res2.history)
print("bibkeys headers:", dict(res2.headers))

search.json status: 200
search.json awal: {"numFound":4063,"start":0,"numFoundExact":true,"num_found":4063,"documentation_url":"https://openlibrary.org/dev/docs/api/search","q":"harry potter","offset":null,"docs":[{"author_key":["OL23919A"],"
bibkeys status: 404
bibkeys url akhir: https://openlibrary.org/api/books?bibkeys=ISBN:9780140328721&format=json&jscmd=data
bibkeys history (redirect): []
bibkeys headers: {'Server': 'nginx/1.30.5', 'Date': 'Wed, 23 Sep 2026 04:15:58 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'access-control-allow-origin': '*', 'access-control-allow-method': 'GET, OPTIONS', 'access-control-max-age': '86400', 'x-ol-stats': '""', 'Content-Encoding': 'gzip'}


In [ ]:
import cv2
print(cv2.__version__)
print(hasattr(cv2, "barcode"))
cv2.barcode.BarcodeDetector()  # kalau ini error, tempel pesannya

5.0.0
True


< cv2.barcode.BarcodeDetector 0x78c8d13f32d0>

## 11. Mode Batch (admin upload banyak cover sekaligus)

.

In [ ]:
def scan_folder_batch(folder_path: str, output_json: str = "hasil_scan_batch.json"):
    image_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ])

    hasil_batch = []
    for fname in image_files:
        fpath = os.path.join(folder_path, fname)
        hasil = ask_model(fpath)
        hasil["nama_file"] = fname
        hasil_batch.append(hasil)
        tanda_cache = " (dari cache)" if hasil.get("_dari_cache_scan") else ""
        print(f"[{fname}]{tanda_cache} judul: {hasil.get('judul')!r} | penerbit: {hasil.get('penerbit')!r}")

    out_path = os.path.join(folder_path, output_json)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(hasil_batch, f, ensure_ascii=False, indent=2)

    print(f"\nTersimpan {len(hasil_batch)} draft ke {out_path}")
    return hasil_batch


# Contoh pakai:
# hasil_draft = scan_folder_batch(FOLDER_PATH)